In [1]:
import pandas as pd
import xgboost as xgb
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

In [2]:
train = pd.read_csv('../001.preprocess/train/train_A.csv')

In [3]:
# 特徴量とターゲットの定義
X = train[['user_id', 'product_id', 'user_interest', 'product_interest', 'event_type']]
y = train['event_type']

In [4]:
# データを訓練用とテスト用に分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(2701180, 5)
(675296, 5)
(2701180,)
(675296,)


In [10]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2701180 entries, 1895889 to 2219121
Data columns (total 5 columns):
 #   Column            Dtype  
---  ------            -----  
 0   user_id           int64  
 1   product_id        int64  
 2   user_interest     float64
 3   product_interest  float64
 4   event_type        int64  
dtypes: float64(2), int64(3)
memory usage: 123.7 MB


In [11]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 675296 entries, 730974 to 1674089
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   user_id           675296 non-null  int64  
 1   product_id        675296 non-null  int64  
 2   user_interest     668692 non-null  float64
 3   product_interest  668692 non-null  float64
 4   event_type        675296 non-null  int64  
dtypes: float64(2), int64(3)
memory usage: 30.9 MB


In [12]:
y_train.info()

<class 'pandas.core.series.Series'>
Index: 2701180 entries, 1895889 to 2219121
Series name: event_type
Non-Null Count    Dtype
--------------    -----
2701180 non-null  int64
dtypes: int64(1)
memory usage: 41.2 MB


In [13]:
y_test.info()

<class 'pandas.core.series.Series'>
Index: 675296 entries, 730974 to 1674089
Series name: event_type
Non-Null Count   Dtype
--------------   -----
675296 non-null  int64
dtypes: int64(1)
memory usage: 10.3 MB


In [ ]:
# XGBoostのデータ形式に変換
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

In [ ]:
# XGBoostのモデル学習
params = {
    'objective': 'multi:softmax',  # 多クラス分類
    'num_class': len(np.unique(y_train)),  # クラス数
    'eval_metric': 'mlogloss',  # ロス関数
    'max_depth': 6,  # 木の深さ
    'eta': 0.1,  # 学習率
    'subsample': 0.8,  # サンプル割合
}

model = xgb.train(params, xgb.DMatrix(X_train, label=y_train), num_boost_round=100)

In [ ]:
# 予測
y_pred = model.predict(xgb.DMatrix(X_test))

In [ ]:
# NDCG評価
y_pred_ranked = np.argsort(-y_pred)  # 降順でソート
ndcg = ndcg_score([y_test], [y_pred_ranked])

In [ ]:
print("NDCG Score: ", ndcg)

In [ ]:
print(y_pred)